# Pure-State Strip Entanglement Contours

Loads saved pure-state covariance snapshots and computes y0-averaged strip entanglement contours on CUDA. This notebook is analysis-only: it does not run Markov dynamics.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'strip_entropy_contours_gpu.py').exists() and (path / 'gpu_data').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with src/strip_entropy_contours_gpu.py and gpu_data')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')

In [ ]:
import gc
from typing import Any

import numpy as np
from tqdm.auto import tqdm

In [ ]:
GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
SOURCE_CAMPAIGN = 'pure_state_covariance_snapshots'
OUTPUT_CAMPAIGN = 'pure_state_strip_entropy_contours'
SOURCE_ROOT = GPU_DATA_ROOT / SOURCE_CAMPAIGN
OUTPUT_ROOT = GPU_DATA_ROOT / OUTPUT_CAMPAIGN
RUNS_ROOT = OUTPUT_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

EPS = 1e-12
AUTOTUNE_Y0_CHUNK = True
Y0_CHUNK_CANDIDATES = [1, 2, 4, 8, 12, 16, 24, 32, 'ny']
AUTOTUNE_REPEAT = 2
SKIP_EXISTING = True
CANONICAL_SOURCE_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
EXPECTED_SNAPSHOT_CYCLES = [5, 10, 20, 50]
EXPECTED_SOURCE_KEYS = {(16, 30, 1), (16, 30, 2), (16, 40, 1), (16, 40, 2)}
EXPECTED_SHARD_SHAPES = {
    (16, 30): (10, 4, 960, 960),
    (16, 40): (10, 4, 1280, 1280),
}

print(json.dumps({
    'source_root': str(SOURCE_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'eps': EPS,
    'autotune_y0_chunk': AUTOTUNE_Y0_CHUNK,
    'y0_chunk_candidates': Y0_CHUNK_CANDIDATES,
    'autotune_repeat': AUTOTUNE_REPEAT,
    'skip_existing': SKIP_EXISTING,
}, indent=2))

In [ ]:
def load_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def write_json_atomic_local(path: Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp_path.replace(path)


def rel_to_gpu_data(path: Path) -> str | None:
    try:
        return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))
    except Exception:
        return None


def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        index = load_json(index_path)
    else:
        index = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = index.setdefault('campaigns', [])
    entry = {
        'name': OUTPUT_CAMPAIGN,
        'kind': 'derived_entropy_observable',
        'manifest': f'{OUTPUT_CAMPAIGN}/campaign_manifest.json',
    }
    campaigns[:] = [c for c in campaigns if c.get('name') != OUTPUT_CAMPAIGN]
    campaigns.append(entry)
    index['bundle_root'] = str(BUNDLE_ROOT)
    index['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic_local(index_path, index)


def validate_expected_source_entry(entry: dict[str, Any], manifest: dict[str, Any], summary: dict[str, Any], shard_path: Path) -> tuple[int, int, int, tuple[int, int, int, int]]:
    nx, ny, nshell = int(entry['Nx']), int(entry['Ny']), int(entry['nshell'])
    key = (nx, ny, nshell)
    if key not in EXPECTED_SOURCE_KEYS:
        raise ValueError(f'Unexpected pure-state snapshot config {key} in campaign manifest.')
    if int(entry.get('samples', -1)) != 10 or int(entry.get('cycles', -1)) != 50:
        raise ValueError(f'Unexpected samples/cycles for {key}: samples={entry.get("samples")}, cycles={entry.get("cycles")}')
    if [int(c) for c in entry.get('snapshot_cycles', [])] != EXPECTED_SNAPSHOT_CYCLES:
        raise ValueError(f'Unexpected snapshot cycles for {key}: {entry.get("snapshot_cycles")}')
    if entry.get('protocol') != 'perfect_correction' or entry.get('init_mode') != 'default':
        raise ValueError(f'Unexpected protocol/init for {key}: {entry.get("protocol")}/{entry.get("init_mode")}')
    if entry.get('dtype_resolved') != 'complex128':
        raise ValueError(f'Unexpected entry dtype for {key}: {entry.get("dtype_resolved")}')
    if summary.get('canonical_dynamics_entry_point') != CANONICAL_SOURCE_ENTRY_POINT:
        raise ValueError(f'Unexpected source entry point in {shard_path.parent / "run_summary.json"}')
    if summary.get('protocol') != 'perfect_correction' or summary.get('init_mode') != 'default':
        raise ValueError(f'Unexpected summary protocol/init for {key}: {summary.get("protocol")}/{summary.get("init_mode")}')
    if summary.get('dtype_resolved') != 'complex128':
        raise ValueError(f'Unexpected summary dtype for {key}: {summary.get("dtype_resolved")}')
    cfg = manifest.get('config', {})
    if manifest.get('store_mode') != 'snapshots' or cfg.get('store_mode') != 'snapshots':
        raise ValueError(f'Expected snapshot store mode for {key}.')
    if manifest.get('num_batches') != 1:
        raise ValueError(f'Expected one shard for {key}, got num_batches={manifest.get("num_batches")}.')
    if cfg.get('dtype') != 'c128':
        raise ValueError(f'Expected manifest dtype c128 for {key}, got {cfg.get("dtype")}.')
    if cfg.get('sequence') != 'raster_y':
        raise ValueError(f"Expected sequence='raster_y' for {key}, got {cfg.get('sequence')!r}.")
    if cfg.get('postselect') is not False or cfg.get('perfect_correction') is not True:
        raise ValueError(f'Expected non-postselected perfect-correction source for {key}.')
    if [int(c) for c in cfg.get('snapshot_cycles', [])] != EXPECTED_SNAPSHOT_CYCLES:
        raise ValueError(f'Unexpected manifest snapshot cycles for {key}: {cfg.get("snapshot_cycles")}')
    shards = manifest.get('shards', {})
    if sorted(int(k) for k in shards) != [0]:
        raise ValueError(f'Expected exactly shard 0 for {key}, got {sorted(shards)}.')
    expected_shape = EXPECTED_SHARD_SHAPES[(nx, ny)]
    if [int(x) for x in shards['0'].get('shape', [])] != list(expected_shape):
        raise ValueError(f'Manifest shard shape mismatch for {key}: {shards["0"].get("shape")} vs {expected_shape}.')
    if not shard_path.exists():
        raise FileNotFoundError(f'Missing source shard for {key}: {shard_path}')
    shard = np.load(shard_path, mmap_mode='r')
    if shard.shape != expected_shape or shard.dtype != np.complex128:
        raise ValueError(f'{shard_path}: expected {expected_shape} complex128, got {shard.shape} {shard.dtype}')
    return nx, ny, nshell, expected_shape


def source_run_records() -> list[dict[str, Any]]:
    campaign_path = SOURCE_ROOT / 'campaign_manifest.json'
    campaign = load_json(campaign_path)
    if campaign.get('canonical_dynamics_entry_point') != CANONICAL_SOURCE_ENTRY_POINT:
        raise ValueError(f'Unexpected campaign entry point in {campaign_path}.')
    if campaign.get('protocol') != 'perfect_correction' or campaign.get('dtype') != 'complex128':
        raise ValueError(f'Unexpected source campaign protocol/dtype in {campaign_path}.')
    if [int(c) for c in campaign.get('snapshot_cycles', [])] != EXPECTED_SNAPSHOT_CYCLES:
        raise ValueError(f'Unexpected campaign snapshot cycles: {campaign.get("snapshot_cycles")}')
    results = campaign.get('results', [])
    if len(results) != len(EXPECTED_SOURCE_KEYS):
        raise ValueError(f'Expected {len(EXPECTED_SOURCE_KEYS)} source configs, found {len(results)}.')

    records = []
    seen = set()
    for entry in results:
        run_dir = GPU_DATA_ROOT / entry['run_dir_relative']
        manifest_path = GPU_DATA_ROOT / entry['manifest_path_relative']
        summary_path = GPU_DATA_ROOT / entry['summary_path_relative']
        manifest = load_json(manifest_path)
        summary = load_json(summary_path)
        shard_path = run_dir / manifest['shards']['0']['filename']
        nx, ny, nshell, expected_shape = validate_expected_source_entry(entry, manifest, summary, shard_path)
        key = (nx, ny, nshell)
        if key in seen:
            raise ValueError(f'Duplicate source config {key}.')
        seen.add(key)
        records.append({
            **entry,
            'run_dir': run_dir,
            'manifest_path': manifest_path,
            'summary_path': summary_path,
            'shard_path': shard_path,
            'source_manifest': manifest,
            'source_summary': summary,
            'expected_shard_shape': expected_shape,
        })

    missing = EXPECTED_SOURCE_KEYS - seen
    extra = seen - EXPECTED_SOURCE_KEYS
    if missing or extra:
        raise ValueError(f'Source config mismatch: missing={sorted(missing)}, extra={sorted(extra)}')
    return sorted(records, key=lambda rec: (int(rec['Ny']), int(rec['nshell'])))


SOURCE_RUNS = source_run_records()
for record in SOURCE_RUNS:
    print(f"[source] N{record['Nx']}x{record['Ny']} nsh={record['nshell']} shard={record['shard_path']} shape={record['expected_shard_shape']} cycles={record['snapshot_cycles']}")

In [ ]:
import importlib
import inspect
import torch

import strip_entropy_contours_gpu as strip_entropy_contours_gpu_module
strip_entropy_contours_gpu_module = importlib.reload(strip_entropy_contours_gpu_module)

from strip_entropy_contours_gpu import (
    HELPER_VERSION,
    autotune_y0_chunk_for_ay,
    compute_strip_entropy_for_ay,
    save_npz_atomic,
    validate_saved_strip_product,
    write_json_atomic,
)

helper_source = inspect.getsource(strip_entropy_contours_gpu_module.entropy_contour_batch_torch)
if '_eigh_with_fallback' not in helper_source:
    raise RuntimeError(
        'Loaded strip_entropy_contours_gpu.py does not contain the robust eigh fallback. '
        'Restart the runtime and rerun from the setup cell.'
    )
print(f'[setup] strip_entropy_contours_gpu helper: {strip_entropy_contours_gpu_module.__file__}')
print(f'[setup] strip_entropy_contours_gpu version: {HELPER_VERSION}')

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = 'cuda:0'
DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
if CUDA_AVAILABLE:
    device_props = torch.cuda.get_device_properties(DEVICE)
    total_gib = device_props.total_memory / 1024**3
    print(f'[setup] torch={torch.__version__}')
    print(f'[setup] cuda device: {DEVICE_NAME} ({total_gib:.2f} GiB)')
    if 'A100' not in DEVICE_NAME or total_gib < 35:
        print('[warn] This notebook is sized for an A100 40GB runtime.')
else:
    print('[setup] CUDA is not available. Source validation can run, but computation requires CUDA.')


def output_case_dir(record: dict[str, Any]) -> Path:
    return RUNS_ROOT / f"N{record['Nx']}x{record['Ny']}_nsh{record['nshell']}_{record['protocol']}"


def product_arrays_for_save(result: dict[str, Any], metadata: dict[str, Any]) -> dict[str, Any]:
    arrays = {
        key: value for key, value in result.items()
        if key not in {'validation_metrics'}
    }
    arrays['config_json'] = np.asarray(json.dumps(metadata, sort_keys=True))
    arrays['validation_json'] = np.asarray(json.dumps(result.get('validation_metrics', []), sort_keys=True))
    arrays['autotune_json'] = np.asarray(json.dumps(metadata.get('autotune', {}), sort_keys=True))
    return arrays


def existing_product_metadata(path: Path) -> dict[str, Any] | None:
    required = {'selected_y0_chunk', 'final_y0_chunk', 'autotune_json'}
    with np.load(path, allow_pickle=False) as data:
        if not required.issubset(set(data.files)):
            return None
        return {
            'selected_y0_chunk': int(np.asarray(data['selected_y0_chunk'])),
            'final_y0_chunk': int(np.asarray(data['final_y0_chunk'])),
            'autotune_json': str(np.asarray(data['autotune_json'])),
        }


def zero_ay_autotune_metadata() -> dict[str, Any]:
    return {
        'selected_y0_chunk': 0,
        'candidates': [],
        'trials': [],
        'repeat': AUTOTUNE_REPEAT,
        'reason': 'Ay=0 has empty subsystem width and does not require eigensolve autotuning.',
    }


def select_y0_chunk(record: dict[str, Any], ay: int) -> tuple[int, dict[str, Any]]:
    nx, ny = int(record['Nx']), int(record['Ny'])
    if ay == 0:
        metadata = zero_ay_autotune_metadata()
        return 0, metadata
    if not AUTOTUNE_Y0_CHUNK:
        fallback = min(8, ny)
        metadata = {
            'selected_y0_chunk': fallback,
            'candidates': Y0_CHUNK_CANDIDATES,
            'trials': [],
            'repeat': AUTOTUNE_REPEAT,
            'reason': 'Autotuning disabled; using fallback chunk.',
        }
        return fallback, metadata
    metadata = autotune_y0_chunk_for_ay(
        shard_path=record['shard_path'],
        nx=nx,
        ny=ny,
        ay=ay,
        device=DEVICE,
        eps=EPS,
        candidates=Y0_CHUNK_CANDIDATES,
        repeat=AUTOTUNE_REPEAT,
        validate=True,
        progress=tqdm,
    )
    return int(metadata['selected_y0_chunk']), metadata


def compute_one_record(record: dict[str, Any]) -> dict[str, Any]:
    if not CUDA_AVAILABLE:
        raise RuntimeError('CUDA is required for strip-contour computation; source validation already completed above.')

    nx, ny = int(record['Nx']), int(record['Ny'])
    snapshot_cycles = [int(c) for c in record['snapshot_cycles']]
    case_dir = output_case_dir(record)
    case_dir.mkdir(parents=True, exist_ok=True)
    products = []
    validation = []
    autotune_records = []

    print(f"[config] N{nx}x{ny} nshell={record['nshell']} output={case_dir}")
    ay_iter = tqdm(range(0, ny // 2 + 1), desc=f"Ay N{nx}x{ny} nsh{record['nshell']}", unit='Ay')
    for ay in ay_iter:
        product_dir = case_dir / f'Ay_{ay:03d}'
        product_path = product_dir / f'strip_entropy_contour_Ay{ay:03d}.npz'
        product_rel = rel_to_gpu_data(product_path)
        if SKIP_EXISTING and product_path.exists():
            validate_saved_strip_product(product_path, time_count=len(snapshot_cycles), nx=nx, ay=ay)
            existing_metadata = existing_product_metadata(product_path)
            if existing_metadata is not None:
                products.append({
                    'Ay': int(ay),
                    'product_path': str(product_path),
                    'product_path_relative': product_rel,
                    'selected_y0_chunk': existing_metadata['selected_y0_chunk'],
                    'final_y0_chunk': existing_metadata['final_y0_chunk'],
                    'skipped_existing': True,
                })
                continue
            tqdm.write(f'[recompute] {product_path} is missing autotune metadata; recomputing.')

        selected_y0_chunk, autotune_metadata = select_y0_chunk(record, ay)
        result = compute_strip_entropy_for_ay(
            shard_path=record['shard_path'],
            nx=nx,
            ny=ny,
            ay=ay,
            snapshot_cycles=snapshot_cycles,
            device=DEVICE,
            eps=EPS,
            y0_chunk_max=max(1, selected_y0_chunk),
            validate=True,
            progress=tqdm,
        )
        metadata = {
            'canonical_source_entry_point': CANONICAL_SOURCE_ENTRY_POINT,
            'derived_observable': 'y0-averaged strip entanglement contour',
            'source_campaign': SOURCE_CAMPAIGN,
            'source_run_dir_relative': rel_to_gpu_data(record['run_dir']),
            'source_manifest_relative': rel_to_gpu_data(record['manifest_path']),
            'source_shard_relative': rel_to_gpu_data(record['shard_path']),
            'Nx': nx,
            'Ny': ny,
            'nshell': int(record['nshell']),
            'protocol': record['protocol'],
            'snapshot_cycles': snapshot_cycles,
            'Ay': int(ay),
            'sample_count': int(result['sample_count']),
            'y0_count': int(result['y0_count']),
            'eps': EPS,
            'dtype': 'complex128',
            'device': DEVICE,
            'device_name': DEVICE_NAME,
            'selected_y0_chunk': int(result['selected_y0_chunk']),
            'final_y0_chunk': int(result['final_y0_chunk']),
            'peak_cuda_gib': float(result['peak_cuda_gib']),
            'autotune': autotune_metadata,
            'coordinate_convention': 'local strip coordinates (x, dy) after y0 averaging',
            'product_path_relative': product_rel,
        }
        save_npz_atomic(product_path, **product_arrays_for_save(result, metadata))
        validate_saved_strip_product(product_path, time_count=len(snapshot_cycles), nx=nx, ay=ay)
        product_record = {
            'Ay': int(ay),
            'product_path': str(product_path),
            'product_path_relative': product_rel,
            'selected_y0_chunk': int(result['selected_y0_chunk']),
            'final_y0_chunk': int(result['final_y0_chunk']),
            'peak_cuda_gib': float(result['peak_cuda_gib']),
            'skipped_existing': False,
        }
        products.append(product_record)
        autotune_records.append({
            'Ay': int(ay),
            'selected_y0_chunk': int(autotune_metadata.get('selected_y0_chunk', result['selected_y0_chunk'])),
            'selected_mean_ms': autotune_metadata.get('selected_mean_ms'),
            'selected_peak_cuda_gib': autotune_metadata.get('selected_peak_cuda_gib'),
            'trial_count': len(autotune_metadata.get('trials', [])),
        })
        validation.append({
            'Ay': int(ay),
            'max_total_mismatch': max((m['max_total_mismatch'] for m in result.get('validation_metrics', [])), default=0.0),
            'min_occupation_eval': min((m['min_occupation_eval'] for m in result.get('validation_metrics', [])), default=0.0),
            'max_occupation_eval': max((m['max_occupation_eval'] for m in result.get('validation_metrics', [])), default=0.0),
            'eigh_cpu_fallback_count': sum((int(m.get('eigh_cpu_fallback_count', 0)) for m in result.get('validation_metrics', []))),
        })
        del result
        gc.collect()
        torch.cuda.empty_cache()

    computed = [p for p in products if not p.get('skipped_existing')]
    selected_chunks = [int(p['selected_y0_chunk']) for p in products if int(p.get('Ay', 0)) > 0]
    final_chunks = [int(p['final_y0_chunk']) for p in products if int(p.get('Ay', 0)) > 0]
    summary = {
        'derived_observable': 'y0-averaged strip entanglement contour',
        'source_campaign': SOURCE_CAMPAIGN,
        'source_run_dir_relative': rel_to_gpu_data(record['run_dir']),
        'source_manifest_relative': rel_to_gpu_data(record['manifest_path']),
        'source_shard_relative': rel_to_gpu_data(record['shard_path']),
        'Nx': nx,
        'Ny': ny,
        'nshell': int(record['nshell']),
        'protocol': record['protocol'],
        'samples': int(record['samples']),
        'snapshot_cycles': snapshot_cycles,
        'Ay_values': list(range(0, ny // 2 + 1)),
        'eps': EPS,
        'dtype': 'complex128',
        'device': DEVICE,
        'device_name': DEVICE_NAME,
        'autotune_enabled': AUTOTUNE_Y0_CHUNK,
        'autotune_candidates': Y0_CHUNK_CANDIDATES,
        'autotune_repeat': AUTOTUNE_REPEAT,
        'autotune_records': autotune_records,
        'selected_y0_chunk_min': min(selected_chunks) if selected_chunks else 0,
        'selected_y0_chunk_max': max(selected_chunks) if selected_chunks else 0,
        'final_y0_chunk_min': min(final_chunks) if final_chunks else 0,
        'final_y0_chunk_max': max(final_chunks) if final_chunks else 0,
        'max_peak_cuda_gib': max((float(p.get('peak_cuda_gib', 0.0)) for p in computed), default=0.0),
        'products': products,
        'validation_summary': validation,
        'case_dir_relative': rel_to_gpu_data(case_dir),
    }
    summary_path = case_dir / 'run_summary.json'
    summary['summary_path'] = str(summary_path)
    summary['summary_path_relative'] = rel_to_gpu_data(summary_path)
    write_json_atomic(summary_path, summary)
    return summary

In [ ]:
if not CUDA_AVAILABLE:
    raise RuntimeError('CUDA is required to compute strip entropy products. Source loading and validation completed successfully above.')

all_results = []
for record in tqdm(SOURCE_RUNS, desc='Configs', unit='config'):
    all_results.append(compute_one_record(record))

campaign_manifest = {
    'derived_observable': 'y0-averaged strip entanglement contour',
    'source_campaign': SOURCE_CAMPAIGN,
    'source_campaign_manifest_relative': f'{SOURCE_CAMPAIGN}/campaign_manifest.json',
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_gpu_data(OUTPUT_ROOT),
    'runs_root_relative': rel_to_gpu_data(RUNS_ROOT),
    'eps': EPS,
    'autotune_enabled': AUTOTUNE_Y0_CHUNK,
    'autotune_candidates': Y0_CHUNK_CANDIDATES,
    'autotune_repeat': AUTOTUNE_REPEAT,
    'device': DEVICE,
    'device_name': DEVICE_NAME,
    'dtype': 'complex128',
    'results': all_results,
}
campaign_manifest_path = OUTPUT_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, campaign_manifest)
update_gpu_data_index()
print(f'[done] campaign manifest: {campaign_manifest_path}')

In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')
